# Thinking Out Loud: within-session ML, ensemble, and feature-space exploration

Standalone Kaggle notebook. Each subject and recording session is evaluated independently using nested CV. The four inner-speech direction labels remain the prediction target. The derivative loader preserves the audited cue-relative [0.5, 3.0) interval. No previous notebook, saved splits, or feature exports are needed.

**Default features:** the eight-window implementation from your supplied CUDA reference: Burg AR coefficients, normalized wavelet entropy, and log wavelet variance, using `db2` at DWT level 3. These are three descriptor families, not three different wavelet families. Each trial remains one row. The extractor is adapted to the independently audited time interval rather than copying the reference's different [1.0, 3.5) FIF crop.

`FEATURE_SET='bandpower'` restores the earlier 640-column Welch representation. `eight_window` yields 12,288 columns at 128 channels, AR order 4 and DWT level 3. `combined` reproduces the reference's fusion with six filtered-band power values for the whole epoch and three temporal bins, yielding 15,360 columns; it is not concatenation with the old five-band Welch features.

The reference wavelet pipeline filters each action epoch to 2–45 Hz, converts to microvolts, resamples 256→128 Hz, and divides 320 samples into eight 40-sample windows. Symmetric DWT boundary coefficients are retained. Each channel/window supplies 4 AR, 4 entropy, and 4 variance values. This is a specified adaptation, not an exact reproduction of the cited paper's wavelet-packet method.

**Five models:** linear SVM, logistic regression, shrinkage LDA, random forest, and Extra Trees. Each has its own inner-CV search. The ensemble uses all enabled members, with equal hard votes by default; it never chooses members or weights from outer-test scores.

**Stages:** (1) full-channel baseline; (2) optional complete-procedure shuffled-label controls; (3) optional channel reduction. DL/pretrained-model comparison remains the next research stage after this ML reference is established; no unspecified pretrained model is silently selected here.

Start with the default quick run on subject 3, whose sessions have not yet been reviewed under this new model comparison. The quick grid checks the workflow. Use the research preset and all subjects for the full comparison, including subjects 1–2 because the model roster is now different. The notebook writes its own output folder and includes every configured session, including low-scoring ones.


## 1. Dependencies
Missing MNE, PyWavelets and MRMR packages are installed when Kaggle Internet is available. Installed packages are not upgraded by this cell. NumPy 2.x is required for the retained bandpower option. All models run on CPU. The MRMR selector uses `mrmr-selection==0.2.8`, matching your completed experiment.


In [ ]:
import importlib.util
import subprocess
import sys

for module, package in [('mne','mne'), ('pywt','PyWavelets>=1.6,<2'), ('mrmr','mrmr-selection==0.2.8')]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', package])

import numpy as np
if not hasattr(np, 'trapezoid'):
    raise RuntimeError('Use NumPy >= 2.0, matching your previous Kaggle environment.')


## 2. One configuration cell

Edit this cell, then rerun it and the cells below it.

| Setting | What to change |
|---|---|
| `SUBJECTS`, `SESSIONS` | Subject list and sessions; `None` means all available sessions. |
| `SEARCH_PRESET` | `quick` for a small initial grid, `research` for the full reference grid. |
| `K_VALUES`, `MODEL_GRIDS` | Feature counts and each model's hyperparameter lists. |
| `FEATURE_SET`, `WAVELET`, `WAVELET_LEVEL`, `AR_ORDER` | Eight-window extraction, optional fusion, or the earlier bandpower reference. |
| `MRMR_PREFILTER_K` | Training-only ANOVA candidate screening before MRMR; `None` disables screening. |
| `MODELS`, `SELECTORS` | Enable any subset of the five models and ANOVA/MRMR. |
| `RUN_ENSEMBLE` | Train members as usual, then evaluate their fixed voting combination. |
| `CV_SEEDS` | One seed initially; later a declared list such as `[42, 43, 44]`. Report all seeds, not the best. |
| `RUN_BASELINE`, `RUN_PERMUTATION_CONTROL`, `RUN_CHANNEL_REDUCTION` | Enable stages in order. Controls and reduction are off initially. |
| Plot settings | Session, outer fold, feature count, 2D/3D PCA, and whether held-out points are visible. |

Scientific changes create distinct experiment identities. Changing the subject list, stage switches, number of controls, channel-budget list, or seed list can reuse matching completed tasks within the same scientific configuration. Changing model grids, selectors, or voting settings starts a new experiment. To resume across Kaggle sessions, restore the saved output folder at `OUTPUT_ROOT`.

Hard voting supports all five models. Soft voting is available only when every enabled model supports `predict_proba`; remove `linear_svm` to use it. Fixed weights must be declared before inspecting test scores. Ensemble ties use the smallest class ID, consistently with sorted-label majority voting.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, FileLink

# Data and scope
DATA_ROOT = Path('/kaggle/input/datasets/truthisneverlinear/inner-speech-recognition/inner-speech-recognition/derivatives')
OUTPUT_ROOT = Path('/kaggle/working/session_ensemble_results')
SUBJECTS = [3]                         # Full comparison: list(range(1, 11))
SESSIONS = None                        # All sessions; or [1] for one session.
CV_SEEDS = [42]                        # Later: [42, 43, 44]; retain every seed's results.
OUTER_N_SPLITS = 5
INNER_N_SPLITS = 5
ACTION_WINDOW_MNE = (0.5, 3.0)
ACTION_ENDPOINT = 'exclusive'

# Reference eight-window descriptors: Burg AR + DWT entropy + DWT log variance.
FEATURE_SET = 'eight_window'           # 'bandpower', 'eight_window', or 'combined'.
N_WINDOWS = 8
WAVELET = 'db2'
WAVELET_LEVEL = 3
AR_ORDER = 4
FEATURE_SFREQ = 128.0
WAVELET_BANDPASS = (2.0, 45.0)
MRMR_PREFILTER_K = 512                 # ANOVA-screened MRMR; None ranks every raw feature.

# Search settings: lists below remain editable after choosing a preset.
SEARCH_PRESET = 'quick'                # 'quick' or 'research'
PRESETS = {
    'quick': dict(k=[20, 40], C=[0.01, 0.1], trees=100, depth=[8], leaf=[2], shrinkage=['auto']),
    'research': dict(k=[20, 40, 80, 120], C=[0.001, 0.01, 0.1, 1.0],
                     trees=200, depth=[None, 8], leaf=[1, 2], shrinkage=['auto', 0.1, 0.5]),
}
assert SEARCH_PRESET in PRESETS
settings = PRESETS[SEARCH_PRESET]
K_VALUES = settings['k'].copy()
C_VALUES = settings['C'].copy()
SELECTORS = ['anova', 'mrmr']
MODELS = ['linear_svm', 'logistic', 'lda', 'random_forest', 'extra_trees']
MODEL_GRIDS = {
    'linear_svm': {'C': C_VALUES},
    'logistic': {'C': C_VALUES},
    'lda': {'shrinkage': settings['shrinkage']},
    'random_forest': {'n_estimators': [settings['trees']], 'max_depth': settings['depth'],
                      'min_samples_leaf': settings['leaf'], 'max_features': ['sqrt']},
    'extra_trees': {'n_estimators': [settings['trees']], 'max_depth': settings['depth'],
                    'min_samples_leaf': settings['leaf'], 'max_features': ['sqrt']},
}

# Fixed ensemble
RUN_ENSEMBLE = True
ENSEMBLE_METHOD = 'hard'
ENSEMBLE_WEIGHTS = None                # Equal votes; or a fixed list in MODELS order.

# Run in this order: baseline -> controls -> channel reduction.
RUN_BASELINE = True
RUN_PERMUTATION_CONTROL = False
N_PERMUTATIONS = 5                     # Initial diagnostics, not a formal significance claim.
RUN_CHANNEL_REDUCTION = False
CHANNEL_BUDGETS = [128, 64, 32, 16, 8]

# Runtime and persistence
N_JOBS = 1                            # Start conservatively on Kaggle.
SEARCH_VERBOSE = 2                    # Prints completed candidate-fit timings.
RESUME = True
SAVE_FITTED_MODELS = False             # True enables later model reuse; larger ZIPs.

# Feature-space preview: selected and scaled features -> training-fitted PCA.
MAKE_FEATURE_PLOTS = True
PLOT_SESSION = None                   # First available session, or an explicit integer.
PLOT_OUTER_FOLD = 1
FEATURE_PLOT_K = 40
FEATURE_PLOT_DIMS = 2                 # 2 or 3.
PLOT_INCLUDE_TEST = False             # Optional held-out crosses; do not tune from their appearance.
CLASS_NAMES = {0: 'Up', 1: 'Down', 2: 'Right', 3: 'Left'}

assert ACTION_ENDPOINT == 'exclusive'
assert ACTION_WINDOW_MNE == (0.5, 3.0)
CONFIG = dict(data_root=str(DATA_ROOT), output_root=str(OUTPUT_ROOT), subjects=SUBJECTS,
    sessions=SESSIONS, seeds=CV_SEEDS, action_window_mne=list(ACTION_WINDOW_MNE), endpoint=ACTION_ENDPOINT,
    outer_n_splits=OUTER_N_SPLITS, inner_n_splits=INNER_N_SPLITS,
    feature_set=FEATURE_SET, n_windows=N_WINDOWS, wavelet=WAVELET, wavelet_level=WAVELET_LEVEL,
    ar_order=AR_ORDER, feature_sfreq=FEATURE_SFREQ, wavelet_bandpass=list(WAVELET_BANDPASS),
    mrmr_prefilter_k=MRMR_PREFILTER_K,
    k_values=K_VALUES, selectors=SELECTORS, models=MODELS, model_grids=MODEL_GRIDS,
    run_ensemble=RUN_ENSEMBLE, ensemble_method=ENSEMBLE_METHOD, ensemble_weights=ENSEMBLE_WEIGHTS,
    run_baseline=RUN_BASELINE, run_controls=RUN_PERMUTATION_CONTROL, n_permutations=N_PERMUTATIONS,
    run_channel_reduction=RUN_CHANNEL_REDUCTION, channel_budgets=CHANNEL_BUDGETS,
    n_jobs=N_JOBS, search_verbose=SEARCH_VERBOSE, resume=RESUME, save_models=SAVE_FITTED_MODELS,
    plot_session=PLOT_SESSION, plot_fold=PLOT_OUTER_FOLD, plot_k=FEATURE_PLOT_K,
    plot_dims=FEATURE_PLOT_DIMS, plot_include_test=PLOT_INCLUDE_TEST, class_names=CLASS_NAMES)
print('Preset:', SEARCH_PRESET, '| subjects:', SUBJECTS, '| models:', MODELS)
print('Feature counts:', K_VALUES, '| seeds:', CV_SEEDS)
print('Feature set:', FEATURE_SET, '| MRMR ANOVA candidate pool:', MRMR_PREFILTER_K)
display(pd.DataFrame([dict(model=name, grid=MODEL_GRIDS[name]) for name in MODELS]))


## 3. Embedded pipeline
This notebook contains its complete source. The importable helper module enables reliable caching of transforms on identical fitting data, labels, and parameters. No previously created project file is needed. The executed source is exported with results.

For each subject/session/outer fold/selector, every classifier is tuned on the same inner folds and refitted on outer training. The ensemble votes using those fitted models. Its weights are fixed, and it has no selected inner score because no separate ensemble search is performed. Outer-test predictions cannot affect parameter or member selection.


In [ ]:
CORE_SOURCE = r'''
"""Independent within-session EEG evaluation, with a matched pooled comparator."""
from pathlib import Path
from datetime import datetime, timezone
from time import perf_counter
from zipfile import ZipFile, ZIP_DEFLATED
from importlib.metadata import version, PackageNotFoundError
import hashlib
import json
import warnings
import traceback
import platform
import numpy as np
import pandas as pd
import scipy
import sklearn
from scipy.signal import welch
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_selection import SelectKBest, f_classif, VarianceThreshold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, GridSearchCV, ParameterGrid
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
from threadpoolctl import threadpool_limits

CORE_VERSION = 'within-session-1.0'
BANDS = ((1.,4.), (4.,8.), (8.,13.), (13.,30.), (30.,45.))

def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, default=str,
                                    separators=(',', ':')).encode()).hexdigest()

def array_digest(value):
    value = np.ascontiguousarray(value)
    h = hashlib.sha256(str((value.shape, value.dtype.str)).encode())
    h.update(memoryview(value).cast('B'))
    return h.hexdigest()

def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(value, indent=2, default=str), encoding='utf-8')
    tmp.replace(path)

def environment():
    result = {'python': platform.python_version(), 'numpy': np.__version__,
              'pandas': pd.__version__, 'scipy': scipy.__version__,
              'sklearn': sklearn.__version__}
    for name in ('mne', 'mrmr-selection', 'joblib'):
        try:
            result[name] = version(name)
        except PackageNotFoundError:
            result[name] = 'not installed'
    return result

def bandpower_features(X, sfreq):
    n = min(int(round(sfreq)), X.shape[-1])
    if sfreq / 2 < 45 or n < 16:
        raise ValueError('Insufficient sampling rate or epoch length for declared bands.')
    freqs, density = welch(X, fs=sfreq, window='hann', nperseg=n,
                          noverlap=n//2, detrend='constant', scaling='density', axis=-1)
    values = []
    for low, high in BANDS:
        mask = (freqs >= low) & (freqs <= high)
        if mask.sum() < 2:
            raise ValueError('Insufficient frequency bins.')
        power = np.trapezoid(density[...,mask], freqs[mask], axis=-1)
        values.append(np.log10(np.maximum(power, 1e-24)))
    return np.stack(values, axis=-1).reshape(len(X), -1).astype(np.float64)

def prepare_subject(subject, config):
    import mne
    root = Path(config['data_root'])
    files = sorted(root.rglob(f'sub-{subject:02d}_ses-*_eeg-epo.fif'))
    if not files:
        raise FileNotFoundError(f'No EEG epoch files for subject {subject} below {root}')
    matrices, records, source = [], [], []
    reference = None
    seen = set()
    for path in files:
        session = int(path.name.split('_')[1].split('-')[1])
        if config['sessions'] is not None and session not in config['sessions']:
            continue
        if session in seen:
            raise ValueError(f'Duplicate session {session}: select a single dataset version.')
        seen.add(session)
        event_path = path.with_name(path.name.replace('_eeg-epo.fif', '_events.dat'))
        # Public dataset event files are pickles; use only your trusted dataset copy.
        raw_events = np.asarray(pd.read_pickle(event_path))
        epochs = mne.read_epochs(path, preload=False, proj=False, verbose='ERROR')
        if raw_events.shape != (len(epochs), 4):
            raise ValueError(f'Event dimensions do not match epochs: {path.name}')
        if not np.isfinite(raw_events).all() or not np.equal(raw_events, np.floor(raw_events)).all():
            raise ValueError('Event values must be finite integers.')
        events = raw_events.astype(np.int64)
        if not np.array_equal(events[:,0], epochs.events[:,0]) or not np.array_equal(events[:,1]+31, epochs.events[:,2]):
            raise ValueError('Source event/epoch alignment failed.')
        if not np.isin(events[:,1], [0,1,2,3]).all():
            raise ValueError('Unexpected direction label.')
        original_rows = np.asarray(epochs.selection)
        selected = np.flatnonzero(events[:,2] == 1)
        epochs = epochs[selected].load_data()
        epochs.pick(mne.pick_types(epochs.info, eeg=True, exclude=[]))
        if len(epochs.ch_names) != 128:
            raise ValueError('Exactly 128 EEG channels are required.')
        start, end = config['action_window_mne']
        mask = (epochs.times >= start) & (epochs.times < end)
        times, sfreq, names = epochs.times[mask], float(epochs.info['sfreq']), list(epochs.ch_names)
        if len(times) == 0:
            raise ValueError('Empty action window.')
        if reference is not None and (names != reference[0] or sfreq != reference[1] or not np.array_equal(times,reference[2])):
            raise ValueError('Channels, sampling, or epoch times differ across sessions.')
        reference = names, sfreq, times
        X = epochs.get_data(copy=False)[:,:,mask].astype(np.float32)
        if not np.isfinite(X).all():
            raise ValueError('Nonfinite EEG values.')
        matrix, feature_schema, feature_info = extract_trial_features(X, sfreq, names, config)
        matrices.append(matrix)
        for row, event in zip(original_rows[selected], events[selected]):
            records.append(dict(subject=int(subject), session=session, trial_index=int(row),
                cue_sample_raw=int(event[0]), class_id=int(event[1]),
                trial_id=f'sub-{subject:02d}_ses-{session:02d}_row-{row:04d}'))
        source.append(dict(eeg=str(path), events=str(event_path), inner_trials=len(X),
            X_sha256=array_digest(X), sfreq=sfreq, n_times=len(times),
            first_time=float(times[0]), last_time=float(times[-1]),
            RMS_uV_median=float(np.median(np.sqrt(np.mean(X.astype(float)**2, axis=(1,2)))*1e6))))
        print(f'Subject {subject:02d}, session {session:02d}: {len(X)} trials, {len(names)} channels, {len(times)} samples', flush=True)
    if not matrices:
        raise ValueError('No sessions matched SESSIONS.')
    if config['sessions'] is not None and seen != set(config['sessions']):
        raise ValueError(f'Requested sessions missing: {set(config["sessions"])-seen}')
    metadata = pd.DataFrame(records)
    F = np.concatenate(matrices)
    if metadata.trial_id.duplicated().any() or not np.isfinite(F).all():
        raise ValueError('Duplicate IDs or nonfinite features.')
    return dict(subject=int(subject), F=F, y=metadata.class_id.to_numpy(dtype=np.int64),
                trial_metadata=metadata, ch_names=names, provenance=source, feature_schema=feature_schema, feature_extraction=feature_info)

def stratified_parts(indices, y, n_splits, seed):
    indices = np.asarray(indices, dtype=int)
    counts = np.bincount(y[indices], minlength=4)
    if (counts < n_splits).any():
        raise ValueError(f'Class counts {counts.tolist()} cannot support {n_splits} folds. No automatic fold-count change.')
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return [(indices[a], indices[b]) for a,b in cv.split(np.zeros(len(indices)),y[indices])]

def make_plan(data, config):
    y, metadata = data['y'], data['trial_metadata']
    session_values = sorted(metadata.session.unique().tolist())
    per_session = {s: stratified_parts(np.flatnonzero(metadata.session.to_numpy()==s),y,
                    config['outer_n_splits'], config['seed']) for s in session_values}
    tasks = []
    for fold in range(config['outer_n_splits']):
        inner_by_session = {}
        for s in session_values:
            train, test = per_session[s][fold]
            inner = stratified_parts(train,y,config['inner_n_splits'],config['seed']+fold+1)
            inner_by_session[s] = inner
            tasks.append(dict(mode='within_session', session=int(s), outer_fold=fold+1,
                train=train.tolist(), test=test.tolist(),
                inner=[dict(train=a.tolist(), validation=b.tolist()) for a,b in inner]))
        if config['run_pooled_comparator']:
            tasks.append(dict(mode='pooled_sessions_matched', session=0, outer_fold=fold+1,
                train=np.concatenate([per_session[s][fold][0] for s in session_values]).tolist(),
                test=np.concatenate([per_session[s][fold][1] for s in session_values]).tolist(),
                inner=[dict(train=np.concatenate([inner_by_session[s][j][0] for s in session_values]).tolist(),
                            validation=np.concatenate([inner_by_session[s][j][1] for s in session_values]).tolist())
                       for j in range(config['inner_n_splits'])]))
    plan = dict(subject=data['subject'], seed=config['seed'], trial_ids=metadata.trial_id.tolist(),
                labels=y.tolist(), outer_n_splits=config['outer_n_splits'],
                inner_n_splits=config['inner_n_splits'], tasks=tasks)
    validate_plan(data,plan)
    return plan

def validate_plan(data, plan):
    metadata, y = data['trial_metadata'], data['y']
    if plan['trial_ids'] != metadata.trial_id.tolist() or plan['labels'] != y.tolist():
        raise ValueError('Plan alignment failure.')
    for mode in {t['mode'] for t in plan['tasks']}:
        coverage = np.zeros(len(y),dtype=int)
        for task in [t for t in plan['tasks'] if t['mode']==mode]:
            a,b = set(task['train']),set(task['test'])
            universe = (set(np.flatnonzero(metadata.session.to_numpy()==task['session']))
                        if mode=='within_session' else set(range(len(y))))
            if a & b or a | b != universe or len(a)!=len(task['train']) or len(b)!=len(task['test']):
                raise ValueError('Invalid outer partition.')
            coverage[task['test']] += 1
            inner_coverage = dict.fromkeys(a,0)
            for inner in task['inner']:
                c,d = set(inner['train']),set(inner['validation'])
                if c & d or c | d != a or (c|d)&b or len(c)!=len(inner['train']) or len(d)!=len(inner['validation']):
                    raise ValueError('Invalid inner partition or outer-test leakage.')
                if set(y[list(c)]) != {0,1,2,3} or set(y[list(d)]) != {0,1,2,3}:
                    raise ValueError('Inner fold missing a class.')
                for i in d:
                    inner_coverage[i] += 1
            if set(inner_coverage.values()) != {1}:
                raise ValueError('Inner validation coverage is not exactly once.')
        if not np.all(coverage==1):
            raise ValueError('Outer test coverage is not exactly once per mode.')
    for pooled in [t for t in plan['tasks'] if t['mode']=='pooled_sessions_matched']:
        within = [t for t in plan['tasks'] if t['mode']=='within_session' and t['outer_fold']==pooled['outer_fold']]
        if set(pooled['test']) != {i for t in within for i in t['test']}:
            raise ValueError('Comparison test trials differ.')

class BoundedANOVA(BaseEstimator, TransformerMixin):
    def __init__(self, k=40):
        self.k = k
    def fit(self, X, y):
        self.selector_ = SelectKBest(f_classif,k=min(int(self.k),X.shape[1])).fit(X,y)
        return self
    def transform(self, X):
        return self.selector_.transform(X)
    def get_support(self):
        return self.selector_.get_support()

class FoldMRMR(BaseEstimator, TransformerMixin):
    def __init__(self, k=40):
        self.k = k
    def fit(self, X, y):
        from mrmr import mrmr_classif
        names = [f'f{i}' for i in range(X.shape[1])]
        chosen = mrmr_classif(X=pd.DataFrame(X,columns=names), y=pd.Series(y),
            K=min(int(self.k),X.shape[1]), relevance='f', redundancy='c',
            n_jobs=1, show_progress=False)
        self.selected_ = np.asarray([names.index(n) for n in chosen],dtype=int)
        self.n_features_in_ = X.shape[1]
        if len(self.selected_) == 0:
            raise ValueError('MRMR returned no features.')
        return self
    def transform(self, X):
        return X[:,self.selected_]
    def get_support(self):
        result = np.zeros(self.n_features_in_,dtype=bool)
        result[self.selected_] = True
        return result

"""Eight-window descriptors adapted from the supplied CUDA reference extractor.

AR + normalized DWT entropy + log population variance. Original trials remain rows.
The audited cue-relative interval is supplied by the independent EEG loader.
"""
from fractions import Fraction
from scipy import signal

_base_environment = environment
def environment():
    result = _base_environment()
    try:
        result['PyWavelets'] = version('PyWavelets')
    except PackageNotFoundError:
        result['PyWavelets'] = 'not installed'
    return result

def burg_coefficients(data, order=4):
    x = np.asarray(data, dtype=np.float64)
    if x.ndim != 2 or not np.isfinite(x).all():
        raise ValueError('Burg requires finite [signals, samples].')
    if not isinstance(order, (int,np.integer)) or not 1 <= order < x.shape[1]-1:
        raise ValueError('AR order must be positive and smaller than samples minus one.')
    x = x-x.mean(axis=-1,keepdims=True)
    scale = np.sqrt(np.mean(x*x,axis=-1,keepdims=True))
    x = x/np.maximum(scale,1e-15)
    forward,backward = x[:,1:].copy(),x[:,:-1].copy()
    coefficients = np.empty((len(x),0),dtype=np.float64)
    for lag in range(order):
        denominator = np.sum(forward*forward+backward*backward,axis=-1)
        reflection = np.divide(-2*np.sum(forward*backward,axis=-1),denominator,
            out=np.zeros(len(x)),where=denominator>1e-15)
        reflection = np.clip(reflection,-1+1e-7,1-1e-7)
        coefficients = np.concatenate([
            coefficients+reflection[:,None]*coefficients[:,::-1],reflection[:,None]],axis=1)
        updated_forward = forward+reflection[:,None]*backward
        updated_backward = backward+reflection[:,None]*forward
        forward,backward = updated_forward[:,1:],updated_backward[:,:-1]
    return -coefficients

def wavelet_statistics(coefficients):
    values = np.asarray(coefficients,dtype=np.float64)
    energy = values*values
    total = energy.sum(axis=-1,keepdims=True)
    probability = np.divide(energy,total,out=np.zeros_like(energy),where=total>1e-24)
    entropy = -np.sum(probability*np.log(np.maximum(probability,1e-300)),axis=-1)
    entropy = entropy/np.log(values.shape[-1]) if values.shape[-1]>1 else np.zeros_like(entropy)
    variance = np.log(np.maximum(np.var(values,axis=-1,ddof=0),1e-12))
    return np.clip(entropy,0.,1.),variance

def extract_trial_features(X,sfreq,ch_names,config):
    feature_set = config.get('feature_set','bandpower')
    rows = []
    def add(channel,window,family,component,start,stop):
        rows.append(dict(feature_index=len(rows),channel_index=int(channel),channel=ch_names[channel],
            window=int(window),family=family,component=component,
            start_time_mne=float(config['action_window_mne'][0]+start),
            stop_time_mne=float(config['action_window_mne'][0]+stop),
            feature_name=f'w{window:02d}__{ch_names[channel]}__{family}__{component}'))
    if feature_set=='bandpower':
        F = bandpower_features(X,sfreq)
        for channel in range(len(ch_names)):
            for lo,hi in BANDS:
                add(channel,0,'welch_log10_power',f'{lo:g}-{hi:g}Hz',0,X.shape[-1]/sfreq)
        return F,pd.DataFrame(rows),dict(feature_set=feature_set,units='volts',bands=BANDS,
            features=int(F.shape[1]),one_row_per_original_trial=True)
    if feature_set not in ['eight_window','combined']:
        raise ValueError('FEATURE_SET must be bandpower, eight_window, or combined.')
    import pywt
    n_windows = config.get('n_windows',8)
    wavelet = config.get('wavelet','db2')
    level = config.get('wavelet_level',3)
    ar_order = config.get('ar_order',4)
    target_rate = config.get('feature_sfreq',128.)
    lo,hi = config.get('wavelet_bandpass',[2.,45.])
    if n_windows!=8:
        raise ValueError('This reference implementation uses exactly eight windows.')
    if not 0 < lo < hi < min(sfreq,target_rate)/2:
        raise ValueError('Wavelet bandpass must be below both sampling Nyquist frequencies.')
    ratio = Fraction(float(target_rate)/float(sfreq)).limit_denominator(10000)
    samples = (X.shape[-1]*ratio.numerator+ratio.denominator-1)//ratio.denominator
    minimum_window = samples//n_windows
    maximum = pywt.dwt_max_level(minimum_window,wavelet)
    if not isinstance(level,(int,np.integer)) or not 1<=level<=maximum:
        raise ValueError(f'{wavelet} level {level} exceeds maximum {maximum} for {minimum_window}-sample windows.')
    if not isinstance(ar_order,(int,np.integer)) or not 1<=ar_order<minimum_window-1:
        raise ValueError('AR_ORDER is invalid for the window length.')
    microvolts = np.asarray(X,dtype=np.float32)*1e6
    sos = signal.butter(4,(lo,hi),btype='bandpass',fs=sfreq,output='sos')
    broadband = signal.sosfiltfilt(sos,microvolts,axis=-1).astype(np.float32)
    broadband = signal.resample_poly(broadband,ratio.numerator,ratio.denominator,axis=-1)
    components = [f'A{level}']+[f'D{i}' for i in range(level,0,-1)]
    descriptors = [('ar',f'lag{i}') for i in range(1,ar_order+1)]
    descriptors += [('wavelet_entropy',c) for c in components]
    descriptors += [('wavelet_log_variance',c) for c in components]
    windows = np.array_split(broadband,n_windows,axis=-1)
    matrices=[]
    offset=0
    for window,values in enumerate(windows,1):
        signals = values.reshape(-1,values.shape[-1])
        ar = burg_coefficients(signals,ar_order)
        coefficients = pywt.wavedec(signals.astype(np.float64),wavelet,
                                    mode='symmetric',level=level,axis=-1)
        summaries = [wavelet_statistics(c) for c in coefficients]
        entropy = np.stack([s[0] for s in summaries],axis=-1)
        variance = np.stack([s[1] for s in summaries],axis=-1)
        descriptor_values = np.concatenate([ar,entropy,variance],axis=-1)
        matrices.append(descriptor_values.reshape(len(X),-1))
        for channel in range(len(ch_names)):
            for family,component in descriptors:
                add(channel,window,family,component,offset/target_rate,(offset+values.shape[-1])/target_rate)
        offset += values.shape[-1]
    F = np.concatenate(matrices,axis=1)
    if feature_set=='combined':
        # Reference fusion: six filtered bands, whole epoch + three contiguous temporal bins.
        fusion_bands = ((2.,4.),(4.,8.),(8.,13.),(13.,20.),(20.,30.),(30.,45.))
        filtered = np.stack([signal.sosfiltfilt(signal.butter(4,band,btype='bandpass',fs=sfreq,output='sos'),
                                              microvolts,axis=-1).astype(np.float32) for band in fusion_bands],axis=1)
        edges = np.linspace(0,X.shape[-1],4).astype(int)
        bounds = [(0,X.shape[-1])]+list(zip(edges[:-1],edges[1:]))
        power = []
        for temporal,(start,stop) in enumerate(bounds):
            power.append(np.log(np.maximum(np.mean(filtered[...,start:stop]**2,axis=-1),1e-12)).reshape(len(X),-1))
            for low,high in fusion_bands:
                for channel in range(len(ch_names)):
                    add(channel,temporal,'filtered_log_power',f'{low:g}-{high:g}Hz',start/sfreq,stop/sfreq)
        F = np.concatenate([F,*power],axis=1)
    F = F.astype(np.float32)
    schema = pd.DataFrame(rows)
    if F.shape!=(len(X),len(schema)) or not np.isfinite(F).all() or schema.feature_name.duplicated().any():
        raise ValueError('Invalid descriptor values or schema.')
    info = dict(feature_set=feature_set,n_windows=n_windows,wavelet=wavelet,wavelet_level=level,
        wavelet_components=components,wavelet_mode='symmetric',boundary_coefficients='retained',
        ar_order=ar_order,ar_method='univariate_burg_demeaned',bandpass=[lo,hi],input_units='microvolts',
        input_sfreq=sfreq,feature_sfreq=target_rate,window_lengths=[w.shape[-1] for w in windows],
        entropy='normalized_negative_sum_p_log_p',variance='natural_log_population_variance_floor_1e-12',
        features=F.shape[1],one_row_per_original_trial=True,paper_exact_reproduction=False)
    return F,schema,info

def schema_for(data):
    if 'feature_schema' in data:
        return data['feature_schema']
    # Compatibility with bandpower fixtures and previously exported five-band data.
    rows=[]
    for i in range(data['F'].shape[1]):
        channel=i//5
        rows.append(dict(feature_index=i,channel_index=channel,channel=data['ch_names'][channel],
                         family='welch_log10_power',component=f'{BANDS[i%5][0]}-{BANDS[i%5][1]}Hz',window=0))
    return pd.DataFrame(rows)

class FoldMRMR(BaseEstimator,TransformerMixin):
    """Optional training-only ANOVA screening before the original MRMR procedure."""
    def __init__(self,k=40,prefilter_k=None):
        self.k=k
        self.prefilter_k=prefilter_k
    def fit(self,X,y):
        from mrmr import mrmr_classif
        self.n_features_in_=X.shape[1]
        pool=np.arange(X.shape[1])
        if self.prefilter_k is not None and int(self.prefilter_k)<X.shape[1]:
            scores=f_classif(X,y)[0]
            scores=np.nan_to_num(scores,nan=-np.inf)
            pool=np.sort(np.argsort(-scores,kind='stable')[:int(self.prefilter_k)])
        columns=[f'f{i}' for i in pool]
        chosen=mrmr_classif(X=pd.DataFrame(X[:,pool],columns=columns),y=pd.Series(y),
            K=min(int(self.k),len(pool)),relevance='f',redundancy='c',n_jobs=1,show_progress=False)
        self.selected_=np.array([int(name[1:]) for name in chosen],dtype=int)
        self.pool_indices_=pool
        if len(self.selected_)==0 or not set(self.selected_).issubset(set(pool)):
            raise ValueError('MRMR returned invalid selected columns.')
        return self
    def transform(self,X):
        return X[:,self.selected_]
    def get_support(self):
        result=np.zeros(self.n_features_in_,dtype=bool)
        result[self.selected_]=True
        return result
"""Additional functions for the standalone within-session ML research notebook."""
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.decomposition import PCA
from sklearn.metrics import f1_score
import joblib
import matplotlib.pyplot as plt

CORE_VERSION = 'session-ensemble-wavelet-1.1'
AVAILABLE_MODELS = ('linear_svm', 'logistic', 'lda', 'random_forest', 'extra_trees')

class ChannelBudget(BaseEstimator, TransformerMixin):
    """Rank electrodes on fitting rows only; retain their complete descriptor groups."""
    def __init__(self, count=128, feature_channels=None):
        self.count = count
        self.feature_channels = feature_channels
    def fit(self, X, y):
        if self.feature_channels is None:
            if X.shape[1]%5:
                raise ValueError('Provide the feature-to-channel schema.')
            feature_channels=np.arange(X.shape[1])//5
        else:
            feature_channels=np.asarray(self.feature_channels,dtype=int)
        if len(feature_channels)!=X.shape[1] or feature_channels.min()<0:
            raise ValueError('Feature-to-channel schema does not match the input matrix.')
        self.feature_channels_=feature_channels
        n_channels = int(feature_channels.max()+1)
        if not 1 <= int(self.count) <= n_channels:
            raise ValueError('Requested channel budget is outside the available channel count.')
        if int(self.count) == n_channels:
            self.channels_ = np.arange(n_channels)
        else:
            with warnings.catch_warnings():
                warnings.simplefilter('ignore', RuntimeWarning)
                relevance = f_classif(X, y)[0]
            relevance = np.nan_to_num(relevance, nan=0.0, posinf=1e12, neginf=0.0)
            relevance = np.bincount(feature_channels,weights=relevance,minlength=n_channels)/np.maximum(
                np.bincount(feature_channels,minlength=n_channels),1)
            self.channels_ = np.sort(np.argsort(-relevance, kind='stable')[:int(self.count)])
        self.columns_ = np.flatnonzero(np.isin(feature_channels,self.channels_))
        return self
    def transform(self, X):
        return X[:, self.columns_]

def estimator_for(name, seed):
    if name == 'linear_svm':
        return LinearSVC(dual='auto', max_iter=20000, random_state=seed)
    if name == 'logistic':
        return LogisticRegression(solver='lbfgs', max_iter=5000, random_state=seed)
    if name == 'lda':
        return LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto')
    if name == 'random_forest':
        return RandomForestClassifier(n_jobs=1, random_state=seed)
    if name == 'extra_trees':
        return ExtraTreesClassifier(n_jobs=1, random_state=seed)
    raise ValueError(f'Unknown model: {name}')

def best_index(results):
    scores = np.asarray(results['mean_test_score'])
    if not np.isfinite(scores).all():
        raise ValueError('Nonfinite inner-CV scores. Inspect the model grid.')
    candidates = np.flatnonzero(np.isclose(scores, scores.max(), rtol=0, atol=1e-12))
    return int(min(candidates, key=lambda i: (
        int(results['params'][i]['selector__k']),
        float(results['params'][i].get('model__C', 0)), int(i))))

def model_pipeline(name, selector, channel_count, config, cache):
    return Pipeline([
        ('channels', ChannelBudget(channel_count,config.get('feature_channels'))),
        ('variance', VarianceThreshold(0.0)),
        ('scaler', StandardScaler()),
        ('selector', BoundedANOVA() if selector == 'anova' else FoldMRMR(prefilter_k=config.get('mrmr_prefilter_k'))),
        ('model', estimator_for(name, config['seed'])),
    ], memory=str(cache))

def validate_config(config):
    if not config['subjects'] or len(set(config['subjects'])) != len(config['subjects']) or not set(config['subjects']).issubset(range(1,11)):
        raise ValueError('SUBJECTS must be unique subject IDs from 1 to 10.')
    if config['sessions'] is not None and (not config['sessions'] or len(set(config['sessions'])) != len(config['sessions'])):
        raise ValueError('SESSIONS must be None or a nonempty list of unique session IDs.')
    if not config['selectors'] or not set(config['selectors']).issubset({'anova','mrmr'}):
        raise ValueError('SELECTORS must contain anova and/or mrmr.')
    models = config['models']
    if not models or len(set(models)) != len(models) or not set(models).issubset(AVAILABLE_MODELS):
        raise ValueError(f'MODELS must be unique entries from {AVAILABLE_MODELS}.')
    if config['run_ensemble'] and len(models) < 2:
        raise ValueError('An ensemble requires at least two models.')
    if config['ensemble_method'] not in {'hard','soft'}:
        raise ValueError('ENSEMBLE_METHOD must be hard or soft.')
    if config['run_ensemble'] and config['ensemble_method'] == 'soft' and 'linear_svm' in models:
        raise ValueError('LinearSVC has no predict_proba. Use hard voting, or remove linear_svm for a soft-voting experiment.')
    weights = config['ensemble_weights']
    if weights is not None and (len(weights) != len(models) or not np.isfinite(weights).all() or any(w<0 for w in weights) or sum(weights)<=0):
        raise ValueError('ENSEMBLE_WEIGHTS must contain one nonnegative fixed weight per model, with positive total.')
    if min(config['outer_n_splits'], config['inner_n_splits']) < 2:
        raise ValueError('At least two inner and outer folds are required.')
    if not config['seeds'] or len(set(config['seeds'])) != len(config['seeds']):
        raise ValueError('CV_SEEDS must contain unique integers.')
    if not config['k_values'] or any(int(k)!=k or k<1 for k in config['k_values']):
        raise ValueError('K_VALUES must contain positive integers.')
    for name in models:
        if name not in config['model_grids']:
            raise ValueError(f'Missing MODEL_GRIDS entry for {name}.')
        for key, values in config['model_grids'][name].items():
            if key not in estimator_for(name,config['seeds'][0]).get_params() or not isinstance(values,list) or not values:
                raise ValueError(f'Invalid model grid: {name}.{key}')
    if config['run_channel_reduction'] and (not config['channel_budgets'] or any(int(c)!=c or not 1<=c<=128 for c in config['channel_budgets'])):
        raise ValueError('CHANNEL_BUDGETS must contain integers from 1 through 128.')
    if config['run_controls'] and config['n_permutations'] < 1:
        raise ValueError('N_PERMUTATIONS must be positive when controls are enabled.')
    if not any(config[k] for k in ['run_baseline','run_controls','run_channel_reduction']):
        raise ValueError('Enable at least one training stage.')
    if config.get('feature_set','bandpower') not in ['bandpower','eight_window','combined']:
        raise ValueError('FEATURE_SET must be bandpower, eight_window, or combined.')
    pool=config.get('mrmr_prefilter_k')
    if pool is not None and (int(pool)!=pool or pool<max(config['k_values'])):
        raise ValueError('MRMR_PREFILTER_K must be None or an integer at least as large as max(K_VALUES).')
    if 'mrmr' in config['selectors']:
        from mrmr import mrmr_classif

def shuffled_subject(data, permutation_id, seed):
    """Whole-session label permutation; feature values and original IDs stay fixed."""
    result = dict(data)
    result['trial_metadata'] = data['trial_metadata'].copy()
    result['original_y'] = data['y'].copy()
    y = data['y'].copy()
    rng = np.random.default_rng(seed + 100000*permutation_id + 1000*data['subject'])
    sessions = data['trial_metadata'].session.to_numpy()
    for session in np.unique(sessions):
        ix = np.flatnonzero(sessions == session)
        y[ix] = rng.permutation(y[ix])
    result['y'] = y
    result['trial_metadata']['class_id'] = y
    return result

def vote(member_predictions, weights=None):
    members = np.asarray(member_predictions, dtype=int)
    weights = np.ones(len(members)) if weights is None else np.asarray(weights, dtype=float)
    scores = np.stack([(members==c).T @ weights for c in range(4)], axis=1)
    # Same deterministic tie convention as sorted-label majority voting: smallest class ID.
    return np.argmax(scores, axis=1), scores

def voted_probabilities(models, F, weights=None):
    aligned = []
    for model in models:
        probability = model.predict_proba(F)
        full = np.zeros((len(F),4))
        full[:,np.asarray(model.classes_,dtype=int)] = probability
        aligned.append(full)
    probability = np.average(aligned, axis=0, weights=weights)
    return np.argmax(probability,axis=1), probability

def selected_original_columns(model):
    channel_columns = model.named_steps['channels'].columns_
    retained = channel_columns[model.named_steps['variance'].get_support()]
    return retained[model.named_steps['selector'].get_support()]

def plot_selected_spaces(data, plan, config, output_dir=None):
    """Training-only selection and PCA; optional test projection uses transform only."""
    session = config['plot_session']
    if session is None:
        session = int(data['trial_metadata'].session.min())
    task = next((t for t in plan['tasks'] if t['session']==session and t['outer_fold']==config['plot_fold']),None)
    if task is None:
        raise ValueError('PLOT_SESSION / PLOT_OUTER_FOLD does not match the loaded subject plan.')
    train, test = np.asarray(task['train']), np.asarray(task['test'])
    dims = config['plot_dims']
    if dims not in [2,3]:
        raise ValueError('FEATURE_PLOT_DIMS must be 2 or 3.')
    F,y = data['F'],data['y']
    base = Pipeline([('variance',VarianceThreshold(0.0)),('scaler',StandardScaler())])
    with threadpool_limits(limits=1):
        train_base = base.fit_transform(F[train], y[train])
        test_base = base.transform(F[test])
    matrices = {'before_selection': (train_base,test_base,np.flatnonzero(base.named_steps['variance'].get_support()))}
    for selector in config['selectors']:
        transform = (BoundedANOVA(config['plot_k']) if selector=='anova' else
                     FoldMRMR(config['plot_k'],prefilter_k=config.get('mrmr_prefilter_k')))
        print(f'Feature plot: {selector.upper()} fitted on {len(train)} training trials only',flush=True)
        with threadpool_limits(limits=1):
            transform.fit(train_base,y[train])
        indices = np.flatnonzero(base.named_steps['variance'].get_support())[transform.get_support()]
        matrices[selector] = (transform.transform(train_base),transform.transform(test_base),indices)
    fig = plt.figure(figsize=(6*len(matrices),5.2))
    coordinates, selection_records, projections = [],[],{}
    colors = plt.get_cmap('tab10').colors[:4]
    for position,(name,(A,B,indices)) in enumerate(matrices.items(),start=1):
        if min(A.shape) < dims:
            raise ValueError('Too few rows or selected features for requested PCA dimensions.')
        pca = PCA(n_components=dims,svd_solver='full')
        a,b = pca.fit_transform(A),pca.transform(B)
        projections[name] = dict(train=a,test=b,pca=pca,selected_indices=indices)
        ax = fig.add_subplot(1,len(matrices),position,projection='3d' if dims==3 else None)
        for c in range(4):
            ix = y[train]==c
            kwargs=dict(c=[colors[c]],s=34,alpha=.8,label=config['class_names'].get(c,str(c)))
            if dims==3:
                ax.scatter(a[ix,0],a[ix,1],a[ix,2],**kwargs)
            else:
                ax.scatter(a[ix,0],a[ix,1],**kwargs)
            if config['plot_include_test']:
                ix = y[test]==c
                kwargs=dict(c=[colors[c]],s=65,marker='x',alpha=.95)
                if dims==3:
                    ax.scatter(b[ix,0],b[ix,1],b[ix,2],**kwargs)
                else:
                    ax.scatter(b[ix,0],b[ix,1],**kwargs)
        ax.set_title(f'{name.replace("_"," ").upper()} | {A.shape[1]} features\nPCA variance shown: {pca.explained_variance_ratio_.sum():.1%}')
        ax.set_xlabel('PC1')
        ax.set_ylabel('PC2')
        if dims==3:
            ax.set_zlabel('PC3')
        ax.grid(alpha=.15)
        for split,rows,coords in [('train',train,a),('test',test,b)]:
            for index,point in zip(rows,coords):
                record=dict(subject=data['subject'],session=int(session),outer_fold=config['plot_fold'],
                    selector=name,split=split,trial_id=data['trial_metadata'].trial_id.iloc[index],class_id=int(y[index]))
                record.update({f'PC{i+1}':float(v) for i,v in enumerate(point)})
                coordinates.append(record)
        for i in indices:
            selection_records.append(dict(selector=name,**schema_for(data).iloc[int(i)].to_dict()))
    handles,labels=ax.get_legend_handles_labels()
    fig.legend(handles,labels,loc='lower center',ncol=4)
    fig.suptitle(f"Subject {data['subject']:02d} / Session {session:02d} / Fold {config['plot_fold']} | "
                 + ('training circles; held-out crosses' if config['plot_include_test'] else 'training trials only'))
    fig.tight_layout(rect=[0,.07,1,.95])
    if output_dir is not None:
        directory=Path(output_dir)
        directory.mkdir(parents=True,exist_ok=True)
        stem=f"sub-{data['subject']:02d}_ses-{session:02d}_fold-{config['plot_fold']}_pca-{dims}d"
        fig.savefig(directory/(stem+'.png'),dpi=160)
        pd.DataFrame(coordinates).to_csv(directory/(stem+'_coordinates.csv'),index=False)
        pd.DataFrame(selection_records).to_csv(directory/(stem+'_features.csv'),index=False)
        save_json(directory/(stem+'_projection_info.json'),dict(
            fitted_trial_ids=data['trial_metadata'].trial_id.iloc[train].tolist(),
            test_trial_ids=data['trial_metadata'].trial_id.iloc[test].tolist(),
            k=config['plot_k'],note='Each panel has its own training-fitted PCA axes. Selection uses training labels.'))
    return dict(figure=fig,selected_feature_matrices=matrices,
                feature_space_coordinates=pd.DataFrame(coordinates),projections=projections)

def summarize_results(completed,expected,output_dir,requested_subjects=None):
    folds = pd.DataFrame([r for x in completed for r in x['rows']])
    preds = pd.DataFrame([r for x in completed for r in x['predictions']])
    folds.to_csv(output_dir/'fold_results.csv',index=False)
    preds.to_csv(output_dir/'oof_predictions.csv',index=False)
    summary=[]
    keys=['run_id','stage','subject','session','seed','selector','channel_budget','permutation_id','model']
    for item in expected:
        identity={k:item[k] for k in keys}
        p=preds.loc[np.logical_and.reduce([preds[k].eq(v) for k,v in identity.items()])] if len(preds) else pd.DataFrame()
        r=folds.loc[np.logical_and.reduce([folds[k].eq(v) for k,v in identity.items()])] if len(folds) else pd.DataFrame()
        complete=bool(len(p)==len(item['trial_ids']) and len(p)>0 and not p.trial_id.duplicated().any()
            and set(p.trial_id)==set(item['trial_ids']) and len(r)==item['expected_folds']) if len(p) else False
        summary.append(dict(**identity,complete=complete,completed_folds=len(r),expected_folds=item['expected_folds'],
            predicted_trials=len(p),expected_trials=len(item['trial_ids']),
            test_BA=float(balanced_accuracy_score(p.y_true,p.y_pred)) if complete else np.nan,
            macro_F1=float(f1_score(p.y_true,p.y_pred,average='macro',zero_division=0)) if complete else np.nan,
            train_BA_mean=float(r.train_BA.mean()) if len(r) else np.nan,
            inner_BA_mean=float(r.inner_BA.mean()) if len(r) else np.nan,
            mean_selected_features=float(r.num_selected_features.mean()) if len(r) else np.nan,
            mean_used_channels=float(r.num_used_channels.mean()) if len(r) else np.nan,
            confusion_matrix=json.dumps(confusion_matrix(p.y_true,p.y_pred,labels=[0,1,2,3]).tolist()) if complete else ''))
    summary=pd.DataFrame(summary)
    summary.to_csv(output_dir/'session_summary.csv',index=False)
    # A subject first averages its sessions equally; the overall summary then gives each subject equal weight.
    grouping=['stage','seed','selector','channel_budget','permutation_id','model']
    subject_rows=[]
    if len(summary):
        for group,g in summary.groupby(grouping+['subject']):
            subject_rows.append(dict(zip(grouping+['subject'],group),complete=bool(g.complete.all()),
                n_sessions=len(g),test_BA=float(g.test_BA.mean()) if g.complete.all() else np.nan))
    subject_summary=pd.DataFrame(subject_rows)
    subject_summary.to_csv(output_dir/'subject_summary.csv',index=False)
    cohort=[]
    if len(subject_summary):
        for group,g in subject_summary.groupby(grouping):
            all_complete=bool(g.complete.all() and (requested_subjects is None or set(g.subject)==set(requested_subjects)))
            cohort.append(dict(zip(grouping,group),complete=all_complete,n_subjects=len(g),
                mean_subject_BA=float(g.test_BA.mean()) if all_complete else np.nan,
                std_subject_BA=float(g.test_BA.std(ddof=1)) if all_complete and len(g)>1 else np.nan))
    pd.DataFrame(cohort).to_csv(output_dir/'cohort_summary.csv',index=False)
    return summary,subject_summary,pd.DataFrame(cohort)

def experiment_directory(config):
    source=Path(__file__).read_text(encoding='utf-8')
    # Execution controls do not alter fitted scientific procedures; checkpoint digests handle each task's data/plan.
    exclude={'data_root','output_root','subjects','seeds','run_baseline','run_controls','n_permutations',
             'run_channel_reduction','channel_budgets','resume','n_jobs','search_verbose','save_models',
             'plot_session','plot_fold','plot_k','plot_dims','plot_include_test','class_names'}
    scientific={k:v for k,v in config.items() if k not in exclude}
    identity=dict(core=CORE_VERSION,source_sha256=hashlib.sha256(source.encode()).hexdigest(),
                  scientific_config=scientific,environment=environment())
    directory=Path(config['output_root'])/('experiment-'+digest(identity)[:12])
    return directory,identity,source

def build_row(common,name,model,y_train,train_prediction,y_test,test_prediction,inner_score,elapsed,best_params):
    selected=selected_original_columns(model) if model is not None else np.array([],dtype=int)
    return dict(**common,model=name,train_BA=float(balanced_accuracy_score(y_train,train_prediction)),
        test_BA=float(balanced_accuracy_score(y_test,test_prediction)),inner_BA=inner_score,
        num_selected_features=len(selected) if model is not None else np.nan,
        num_used_channels=len(set(model.named_steps['channels'].feature_channels_[selected])) if model is not None else np.nan,
        requested_k=best_params.get('selector__k') if best_params else None,
        best_params=json.dumps(best_params,default=str,sort_keys=True),fit_seconds=elapsed)

def evaluate_ml_task(data,plan,task,selector,budget,stage,permutation_id,config,root,run_id):
    key=f"{stage}_p{permutation_id}_s{task['session']}_{selector}_c{budget}_f{task['outer_fold']}"
    train,test=np.asarray(task['train']),np.asarray(task['test'])
    F,y,metadata=data['F'],data['y'],data['trial_metadata']
    schema=schema_for(data)
    original_y=data.get('original_y',y)
    checkpoint=root/'checkpoints'/(key+'.json')
    ids=metadata.trial_id.iloc[test].tolist()
    if config['resume'] and checkpoint.exists():
        saved=json.loads(checkpoint.read_text())
        names=config['models']+(['ensemble'] if config['run_ensemble'] else [])
        valid=saved['run_id']==run_id and saved['task_digest']==digest(task) and set(r['model'] for r in saved['rows'])==set(names)
        for name in names:
            p=[p for p in saved['predictions'] if p['model']==name]
            valid=valid and [p['trial_id'] for p in p]==ids and [p['y_true'] for p in p]==y[test].tolist()
        if not valid:
            raise ValueError(f'Invalid checkpoint: {checkpoint}')
        model_files_ready = (not config['save_models'] or all(
            (root/'models'/(key+'_'+name+'.joblib')).exists() for name in config['models']))
        if model_files_ready:
            print(f'Resumed {key}',flush=True)
            return saved
        print(f'Refitting {key} to save requested model files; the saved task had none.',flush=True)
    lookup={int(i):j for j,i in enumerate(train)}
    inner=[(np.array([lookup[i] for i in v['train']]),np.array([lookup[i] for i in v['validation']])) for v in task['inner']]
    common=dict(run_id=run_id,stage=stage,subject=data['subject'],session=int(task['session']),seed=config['seed'],
                selector=selector,channel_budget=int(budget),permutation_id=permutation_id,
                outer_fold=task['outer_fold'],n_train=len(train),n_test=len(test))
    rows,predictions,selection=[],[],[]
    trained,train_preds,test_preds=[],[],[]
    all_warnings=set()
    task_started=perf_counter()
    # Shared cache reuses transforms across classifiers only for identical fitting data, labels and parameters.
    cache=root/'cache'/digest(key)[:10]
    for name in config['models']:
        pipeline=model_pipeline(name,selector,budget,config,cache)
        grid={'selector__k':config['k_values']}
        grid.update({'model__'+k:v for k,v in config['model_grids'][name].items()})
        print(f"\nSubject {data['subject']:02d} / session {task['session']:02d} / seed {config['seed']} / "
              f"{stage} / {selector.upper()} / {budget} channels / fold {task['outer_fold']} / {name}: "
              f"{len(ParameterGrid(grid))} candidates x {len(inner)} inner folds",flush=True)
        search=GridSearchCV(pipeline,grid,scoring='balanced_accuracy',cv=inner,refit=best_index,
            n_jobs=config['n_jobs'],verbose=config['search_verbose'],error_score='raise',return_train_score=True)
        started=perf_counter()
        with warnings.catch_warnings(record=True) as caught,threadpool_limits(limits=1):
            warnings.simplefilter('always')
            search.fit(F[train],y[train])
            model=search.best_estimator_
            tp,ep=model.predict(F[train]),model.predict(F[test])
        elapsed=perf_counter()-started
        all_warnings.update(str(w.message) for w in caught)
        rows.append(build_row(common,name,model,y[train],tp,y[test],ep,
            float(search.cv_results_['mean_test_score'][search.best_index_]),elapsed,search.best_params_))
        for i,p in zip(test,ep):
            predictions.append(dict(**common,model=name,trial_id=str(metadata.trial_id.iloc[i]),
                                    y_true=int(y[i]),original_y_true=int(original_y[i]),y_pred=int(p)))
        selected=selected_original_columns(model)
        selection.append(dict(model=name,selected_feature_indices=selected.tolist(),
            selected_channel_indices=model.named_steps['channels'].channels_.tolist(),
            channels_used_by_selected_features=sorted(set(int(schema.iloc[int(i)].channel_index) for i in selected)),
            selected_features=schema.iloc[selected].to_dict(orient='records')))
        directory=root/'inner_search'
        directory.mkdir(exist_ok=True)
        pd.DataFrame(search.cv_results_).to_csv(directory/(key+'_'+name+'.csv'),index=False)
        if config['save_models']:
            directory=root/'models'
            directory.mkdir(exist_ok=True)
            joblib.dump(model,directory/(key+'_'+name+'.joblib'),compress=3)
        trained.append(model)
        train_preds.append(tp)
        test_preds.append(ep)
        print(f"Finished {name}: train BA={rows[-1]['train_BA']:.3f}, test BA={rows[-1]['test_BA']:.3f}, {elapsed/60:.2f} min",flush=True)
    if config['run_ensemble']:
        if config['ensemble_method']=='hard':
            tp,_=vote(train_preds,config['ensemble_weights'])
            ep,scores=vote(test_preds,config['ensemble_weights'])
        else:
            tp,_=voted_probabilities(trained,F[train],config['ensemble_weights'])
            ep,scores=voted_probabilities(trained,F[test],config['ensemble_weights'])
        members=dict(members=config['models'],method=config['ensemble_method'],fixed_weights=config['ensemble_weights'])
        row=build_row(common,'ensemble',None,y[train],tp,y[test],ep,None,0,members)
        # Feature and channel requirements are the union across members, not their average.
        union={i for s in selection for i in s['selected_feature_indices']}
        row['num_selected_features']=len(union)
        row['num_used_channels']=len({int(schema.iloc[i].channel_index) for i in union})
        rows.append(row)
        for i,p,score in zip(test,ep,scores):
            predictions.append(dict(**common,model='ensemble',trial_id=str(metadata.trial_id.iloc[i]),
                y_true=int(y[i]),original_y_true=int(original_y[i]),y_pred=int(p),vote_scores=json.dumps(score.tolist())))
        print(f"Ensemble ({config['ensemble_method']} voting): test BA={row['test_BA']:.3f}",flush=True)
    saved=dict(run_id=run_id,task_digest=digest(task),rows=rows,predictions=predictions,
               selections=selection,warnings=sorted(all_warnings),task_seconds=perf_counter()-task_started)
    save_json(checkpoint,saved)
    return saved

def export_experiment(directory):
    directory=Path(directory)
    archive=directory.parent/('session_ensemble_review_'+datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')+'.zip')
    expected_path=directory/'expected_results.json'
    active={r['run_id'] for r in json.loads(expected_path.read_text())} if expected_path.exists() else None
    with ZipFile(archive,'w',ZIP_DEFLATED) as z:
        for path in sorted(directory.rglob('*')):
            relative=path.relative_to(directory)
            if relative.parts[0]=='runs' and len(relative.parts)>1 and active is not None and relative.parts[1] not in active:
                continue
            if path.is_file() and 'cache' not in relative.parts and not path.name.endswith('.tmp'):
                z.write(path,relative)
    print('Review ZIP:',archive,flush=True)
    return archive

def run_research(config):
    validate_config(config)
    directory,identity,source=experiment_directory(config)
    directory.mkdir(parents=True,exist_ok=True)
    (directory/'executed_core.py').write_text(source,encoding='utf-8')
    save_json(directory/'configuration.json',config)
    save_json(directory/'environment.json',environment())
    completed,expected,errors=[],[],[]
    names=config['models']+(['ensemble'] if config['run_ensemble'] else [])
    stages=[]
    if config['run_baseline']:
        stages.append(('baseline',0,128))
    if config['run_controls']:
        stages.extend(('label_control',i,128) for i in range(1,config['n_permutations']+1))
    if config['run_channel_reduction']:
        stages.extend(('channel_reduction',0,c) for c in config['channel_budgets'])
    # Declare the complete requested roster before loading so a failed subject is never omitted silently.
    save_json(directory/'requested_roster.json',dict(subjects=config['subjects'],sessions=config['sessions'],
        seeds=config['seeds'],selectors=config['selectors'],models=names,stages=stages))
    try:
        for subject in config['subjects']:
            data_config=dict(config,seed=config['seeds'][0],run_pooled_comparator=False)
            original=prepare_subject(subject,data_config)
            original['feature_schema']=schema_for(original)
            for seed in config['seeds']:
                cfg=dict(config,seed=seed,run_pooled_comparator=False,
                         feature_channels=original['feature_schema'].channel_index.to_numpy())
                for stage,permutation_id,budget in stages:
                    data=shuffled_subject(original,permutation_id,seed) if permutation_id else original
                    plan=make_plan(data,cfg)
                    manifest=dict(**identity,subject=subject,seed=seed,stage=stage,channel_budget=budget,
                        permutation_id=permutation_id,features=array_digest(data['F']),labels=array_digest(data['y']),
                        original_labels=array_digest(original['y']),plan=digest(plan),trial_ids=plan['trial_ids'])
                    run_id=digest(manifest)[:24]
                    root=directory/'runs'/run_id
                    root.mkdir(parents=True,exist_ok=True)
                    save_json(root/'manifest.json',manifest)
                    save_json(root/'plan.json',plan)
                    save_json(root/'input_provenance.json',original['provenance'])
                    save_json(root/'feature_extraction.json',original.get('feature_extraction',{'feature_set':'bandpower'}))
                    original['feature_schema'].to_csv(root/'feature_schema.csv',index=False)
                    data['trial_metadata'].assign(original_class_id=original['y']).to_csv(root/'trial_metadata.csv',index=False)
                    np.savez_compressed(root/'features.npz',F=data['F'],y=data['y'],original_y=original['y'],
                                        trial_ids=np.asarray(plan['trial_ids']),ch_names=np.asarray(data['ch_names']))
                    for session,g in data['trial_metadata'].groupby('session'):
                        for selector in cfg['selectors']:
                            for name in names:
                                expected.append(dict(run_id=run_id,stage=stage,subject=subject,session=int(session),
                                    seed=seed,selector=selector,channel_budget=budget,permutation_id=permutation_id,
                                    model=name,trial_ids=g.trial_id.tolist(),expected_folds=cfg['outer_n_splits']))
                    for task in plan['tasks']:
                        for selector in cfg['selectors']:
                            completed.append(evaluate_ml_task(data,plan,task,selector,budget,stage,permutation_id,cfg,root,run_id))
                            summarize_results(completed,expected,directory,config['subjects'])
    except BaseException as exc:
        errors.append(dict(type=type(exc).__name__,message=str(exc),traceback=traceback.format_exc()))
        raise
    finally:
        summary,subject_summary,cohort_summary=summarize_results(completed,expected,directory,config['subjects'])
        save_json(directory/'expected_results.json',expected)
        save_json(directory/'execution_status.json',dict(errors=errors,subjects_requested=config['subjects'],
            completed_tasks=len(completed),all_requested_completed=bool(len(summary) and summary.complete.all() and not errors),
            note='Models/weights were not selected using outer-test scores. Label controls are exploratory trial-exchangeability diagnostics, not formal significance tests.'))
        export_experiment(directory)
    return dict(output_dir=directory,session_summary=summary,subject_summary=subject_summary,cohort_summary=cohort_summary)

def plot_performances(result,stage='baseline'):
    summary=result['session_summary']
    complete=summary.loc[summary.complete & summary.stage.eq(stage) & summary.permutation_id.eq(0)]
    if complete.empty:
        print('No completed real-label results for this stage.')
        return None
    plot=complete.groupby(['selector','model']).test_BA.mean().unstack('model')
    ax=plot.plot.bar(figsize=(10,4),ylim=(0,1),rot=0)
    ax.axhline(.25,color='black',linestyle='--',linewidth=1,label='Chance reference')
    ax.set_ylabel('Mean session OOF balanced accuracy')
    ax.set_xlabel('Feature selector')
    ax.set_title('All completed sessions; see tables for individual subjects and sessions')
    ax.legend(bbox_to_anchor=(1.02,1),loc='upper left')
    fig=ax.figure
    fig.tight_layout()
    fig.savefig(result['output_dir']/(stage+'_model_comparison.png'),dpi=160)
    return fig
'''

import importlib
module_dir = Path('/kaggle/working/session_ensemble_pipeline_code')
module_dir.mkdir(parents=True, exist_ok=True)
module_path = module_dir / 'session_ensemble_core.py'
module_path.write_text(CORE_SOURCE, encoding='utf-8')
if str(module_dir) not in sys.path:
    sys.path.insert(0, str(module_dir))
import session_ensemble_core as pipeline_core
pipeline_core = importlib.reload(pipeline_core)
pipeline_core.validate_config(CONFIG)
print('Ready:', pipeline_core.CORE_VERSION)
print(pipeline_core.environment())


## 4. Inspect the input and split sizes
The first subject is loaded for exploration; model training has not started. `subject_data`, `F`, `y`, `trial_metadata`, and `plan` remain available for follow-up code.

Features have shape `(original trials, descriptors)`; the exact column-to-channel/window/family mapping is exported. Input epochs retain 128 channels and 640 samples at 256 Hz. Eight-window extraction adds the declared per-trial filtering/resampling, with no additional trial rejection. Insufficient class counts stop the run with an explanation rather than silently changing the fold counts.

The high-dimensional wavelet matrix makes unrestricted MRMR expensive. The default is explicitly **ANOVA-screened MRMR**: within every training fit, retain up to 512 candidates by ANOVA score, then run MRMR. This also applies to preview plots and label controls. Set `MRMR_PREFILTER_K=None` for unrestricted MRMR; it may take substantially longer. Screening is a change to the selector procedure and is recorded in scientific identities.


In [ ]:
preview_config = dict(CONFIG, seed=CV_SEEDS[0], run_pooled_comparator=False)
subject_data = pipeline_core.prepare_subject(SUBJECTS[0], preview_config)
F = subject_data['F']
y = subject_data['y']
trial_metadata = subject_data['trial_metadata']
plan = pipeline_core.make_plan(subject_data, preview_config)

display(trial_metadata.groupby(['subject','session','class_id']).size().unstack('class_id', fill_value=0))
display(pd.DataFrame([
    dict(session=t['session'], fold=t['outer_fold'], train_trials=len(t['train']),
         test_trials=len(t['test']), smallest_inner_train=min(len(i['train']) for i in t['inner']))
    for t in plan['tasks']
]))
print('F:', F.shape, '| y:', y.shape)


In [ ]:
feature_schema = subject_data['feature_schema']
display(feature_schema.groupby('family').size().rename('raw_feature_count'))
display(feature_schema.head(20))
display(pd.Series(subject_data['feature_extraction']))


## 5. Feature-space scatter plots after ANOVA and MRMR
The preview uses a fixed `FEATURE_PLOT_K` for the selected outer fold. It compares standardized descriptors before selection with features after each enabled selector. It is independent of the model grid and is not an accuracy experiment.

Selection and PCA are fitted using **outer-training rows only**. PCA is a display projection; it is not added to the classifiers. Each panel has its own PCA axes and displays its explained variance. Default plots show training trials only. With `PLOT_INCLUDE_TEST=True`, held-out trials are projected with `transform` and drawn as crosses.

Class separation in a training plot is not evidence of generalization: ANOVA/MRMR already used those training labels. Two or three PCs may hide predictive directions. Do not choose k, models, or channels by looking for the best separation of test points. Coordinate CSVs, selected feature names, fit trial IDs, and PNGs are exported.


In [ ]:
experiment_dir, _, _ = pipeline_core.experiment_directory(CONFIG)
if MAKE_FEATURE_PLOTS:
    feature_plot_result = pipeline_core.plot_selected_spaces(
        subject_data, plan, CONFIG, experiment_dir / 'feature_plots')
    selected_feature_matrices = feature_plot_result['selected_feature_matrices']
    feature_space_coordinates = feature_plot_result['feature_space_coordinates']
    plt.show()
    display(feature_space_coordinates.head())
else:
    print('Feature plots disabled.')


## 6. Train the configured stages
This cell runs the baseline, then controls, then reduction if enabled. Progress identifies subject, session, seed, selector, channel budget, fold, and model. Shared caching avoids repeating the same selector fit across model grids. A checkpoint is completed only after all enabled members and the ensemble for that task finish.

With `RESUME=True`, matching completed tasks are reused. Interrupted tasks are retried; their transform caches may still reduce the work. A normal error or keyboard interruption exports completed tasks and a traceback. A forced Kaggle shutdown cannot run the handler, but completed checkpoints remain.

**Label controls:** all labels in one session are shuffled once per permutation; the nested stratified plan is regenerated for the shuffled labels, and both training and test scores use the shuffled target. Every model/selector search and the ensemble are repeated. These are complete-procedure diagnostics under trial-label exchangeability, not the old training-only shuffle. Five permutations are insufficient for a formal significance claim, and temporal dependencies may require a different randomization unit.

**Channel reduction:** channel ranking uses the mean ANOVA F-score across the descriptors assigned to each electrode by the feature schema, fitted inside each inner training fold and outer refit. All descriptors of a retained electrode enter subsequent preprocessing and feature selection. `channel_budget` is the available electrode budget; `num_used_channels` records electrodes actually referenced by selected features. The ensemble's requirement is the union across its members. Different folds may select different channel sets; this is an adaptive channel-selection procedure, not a fixed montage.


In [ ]:
experiment_result = pipeline_core.run_research(CONFIG)
session_summary = experiment_result['session_summary']
subject_summary = experiment_result['subject_summary']
cohort_summary = experiment_result['cohort_summary']
print('Output directory:', experiment_result['output_dir'])
display(session_summary)
display(subject_summary)
display(cohort_summary)


## 7. Compare the five models and ensemble
`test_BA` is computed from all OOF predictions for a subject/session/seed/model/selector/budget. It is blank until that result is complete. `macro_F1` and confusion matrices are also saved. Training scores describe resubstitution performance.

Session summaries preserve every recording. Subject summaries average sessions equally; cohort summaries give each subject equal weight and remain incomplete when requested subjects have not finished. The plot below averages completed sessions for exploration and can differ from the equal-subject cohort average when subjects have different numbers of sessions. Use the tables for the final report. CV folds and repeated seeds are not independent participants; no naive fold t-test is performed.


In [ ]:
pipeline_core.plot_performances(experiment_result, stage='baseline')
plt.show()

baseline = session_summary.loc[
    session_summary.stage.eq('baseline') & session_summary.complete
].copy()
if len(baseline):
    display(baseline.pivot_table(
        index=['subject','session','seed','selector'], columns='model', values='test_BA'))
else:
    print('No complete baseline results in the configured run.')


## 8. Inspect controls and channel-reduction curves
Controls and reduction are optional experiments. They have their own run identities and exported plans. Keep the same declared model/selector procedure when comparing full and reduced channel budgets. Report all declared budgets; selecting a winner after reviewing outer-test curves requires a separate protected evaluation.

The empirical null mean is a diagnostic comparison, not a formal p-value. No arbitrary score threshold automatically approves channel reduction or replaces your research judgment.


In [ ]:
controls = session_summary.loc[
    session_summary.stage.eq('label_control') & session_summary.complete
]
if len(controls):
    display(controls.groupby(['subject','session','seed','selector','model']).test_BA.agg(
        null_mean='mean', null_min='min', null_max='max', permutations='count'))
else:
    print('Label controls were not enabled or have not completed.')

reduction = session_summary.loc[
    session_summary.stage.eq('channel_reduction') & session_summary.complete
]
if len(reduction):
    curve = reduction.groupby(['selector','model','channel_budget']).test_BA.mean().unstack(['selector','model'])
    ax = curve.sort_index(ascending=False).plot(marker='o', figsize=(11, 5), ylim=(0, 1))
    ax.invert_xaxis()
    ax.axhline(0.25, color='black', linestyle='--', linewidth=1)
    ax.set_xlabel('Available EEG channel budget')
    ax.set_ylabel('Mean session OOF balanced accuracy')
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig(experiment_result['output_dir'] / 'channel_reduction_curve.png', dpi=160)
    plt.show()
else:
    print('Channel reduction was not enabled or has not completed.')


## 9. Download the review ZIP
The ZIP includes executed source, configuration, package versions, requested roster, manifests, plans, feature/label arrays, selected channel/band indices, inner-search scores, warnings, fold and OOF results, session/subject/cohort summaries, and saved plots. Optional fitted models are included only when `SAVE_FITTED_MODELS=True`. Cache files are excluded.

This cell can package existing experiment folders after an interruption without retraining. Send the ZIP in our conversation for analysis.

References: [voting ensembles](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.VotingClassifier.html), [PCA](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html), [avoiding leakage](https://scikit-learn.org/stable/common_pitfalls.html), [permutation testing](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.permutation_test_score.html).


In [ ]:
if 'experiment_result' in globals():
    export_dirs = [experiment_result['output_dir']]
else:
    export_dirs = sorted(OUTPUT_ROOT.glob('experiment-*'))
if not export_dirs:
    print('No experiment folder exists yet.')
for directory in export_dirs:
    review_zip = pipeline_core.export_experiment(directory)
    display(FileLink(str(review_zip)))
